# Training: Klasifikasi Sampah
Model EfficientFormer + AugMix + MixUp + ClassBalancedLoss + Progressive

> Jalankan setelah `data_quality.ipynb` (clean & backup) dan `remove_test_leakage.ipynb`.

In [ ]:
import json
import os
import sys
from collections import Counter
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm import tqdm

%matplotlib inline

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

In [ ]:
from modules.utils.config import CLASS_LABELS, IMG_SIZE, MEAN, NUM_CLASSES, PROJECT_ROOT, RESULTS, SEED, STD
from modules.utils.load_data import load_train

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'Project root: {PROJECT_ROOT}')

---
## 1. Load & Split Data
Stratified 80/20 train/val. Filter out 3 file dengan path > 240 chars (gagal dibaca PIL).

In [ ]:
df = load_train()
print(f'Total samples before filter: {len(df)}')

path_col = df['path'].astype(str)
long_mask = path_col.str.len() > 240
print(f'Filtering out {long_mask.sum()} files with path > 240 chars')
df = df[~long_mask].reset_index(drop=True)

print(f'Total samples after filter: {len(df)}')
print(f'\nClass distribution:')
for cls, count in sorted(Counter(df['label']).items()):
    print(f'  {cls}: {count}')

train_df, val_df = train_test_split(
    df, test_size=0.2, stratify=df['label'], random_state=SEED
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print(f'\nTrain: {len(train_df)}, Val: {len(val_df)}')

---
## 2. Dataset & Dataloader
Gunakan `_get_train_transform_v3()` (AugMix + RandomErasing). Opsional MixUp collator.

In [ ]:
_LABEL_TO_IDX = {name: i for i, name in enumerate(CLASS_LABELS)}


def _safe_path(p):
    p = str(p)
    if len(p) > 240 and not p.startswith('\\\\?\\'):
        return '\\\\?\\' + p
    return p


def _open_image(path):
    safe = _safe_path(path)
    with open(safe, 'rb') as f:
        buf = BytesIO(f.read())
    return Image.open(buf).convert('RGB')


class TrashDataset(Dataset):
    def __init__(self, df, transform=None):
        self.paths = (PROJECT_ROOT / df['path']).values
        self.labels = df['label'].map(_LABEL_TO_IDX).values
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = _open_image(self.paths[idx])
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

In [ ]:
from timm.data.auto_augment import augment_ops

aa_params = dict(
    translate_const=int(IMG_SIZE * 0.45),
    img_size=IMG_SIZE,
    magnitude=9,
)
aa_ops = augment_ops('augmix', aa_params)

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.3, 1.0)),
    transforms.RandomHorizontalFlip(),
    aa_ops,
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.3),
])

val_transform = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_ds = TrashDataset(train_df, transform=train_transform)
val_ds = TrashDataset(val_df, transform=val_transform)

train_labels = train_df['label'].map(_LABEL_TO_IDX).values
class_counts = torch.bincount(torch.tensor(train_labels))
class_weights = 1.0 / class_counts.float()
class_weights = class_weights / class_weights.sum() * NUM_CLASSES
val_ds.class_weights = class_weights
print(f'Class (inverse) weights: {class_weights.tolist()}')

# Weighted sampler for imbalance
sample_weights = 1.0 / class_counts[train_labels].float()
sampler = torch.utils.data.WeightedRandomSampler(
    sample_weights, len(sample_weights), replacement=True
)

# MixUp collator
from modules.training.collator import MixUpCollator
USE_MIXUP = True
collate_fn = MixUpCollator(alpha=0.2, num_classes=NUM_CLASSES) if USE_MIXUP else None

BATCH_SIZE = 32
NUM_WORKERS = 4

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
    collate_fn=collate_fn,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
)

print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

---
## 3. Visualisasi Augmentasi
Sample 6 gambar train setelah transform (AugMix + Erasing).

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(15, 5))

for i in range(6):
    # Without augmentation
    img = _open_image(train_ds.paths[i])
    axes[0, i].imshow(img)
    axes[0, i].set_title(f'Original {train_ds.labels[i]}', fontsize=8)
    axes[0, i].axis('off')

    # With augmentation
    aug = train_transform(img)
    aug_np = aug.permute(1, 2, 0).numpy()
    aug_np = aug_np * np.array(STD) + np.array(MEAN)
    aug_np = np.clip(aug_np, 0, 1)
    axes[1, i].imshow(aug_np)
    axes[1, i].set_title(f'Augmented', fontsize=8)
    axes[1, i].axis('off')

plt.suptitle('Dataset Augmentations (AugMix + RandomErasing)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Model & Loss
EfficientFormer-L1 encoder + ClassBalancedLoss.

In [ ]:
from modules.models.factory import TrashClassifier
from modules.training.loss import ClassBalancedLoss

MODEL_NAME = 'efficientformer_l1'  # efficient transformer
model = TrashClassifier(encoder_name=MODEL_NAME, num_classes=NUM_CLASSES)
print(f'Model: {MODEL_NAME}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

criterion = ClassBalancedLoss(beta=0.999, gamma=2.0, num_classes=NUM_CLASSES)
criterion.update(torch.tensor(train_labels))
print(f'CB loss weights: {criterion.weights.tolist()}')

---
## 5. Training
Two-phase: head only → full fine-tune. Progressive disabled (1 stage).

In [ ]:
from modules.training.train import fit

result = fit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    name=f'{MODEL_NAME}_v3',
    encoder_name=MODEL_NAME,
    accumulation_steps=2,
    epochs_head=10,
    epochs_finetune=30,
    lr_head=5e-4,
    lr_finetune=5e-5,
    patience=7,
    criterion=criterion,
    device=DEVICE,
)

print(f'\nBest val F1 macro: {result["best_val_f1"]:.4f}')
print(f'Best epoch: {result["best_epoch"]}')

with open(RESULTS / f'{MODEL_NAME}_v3.json') as f:
    saved = json.load(f)
print(json.dumps(saved, indent=2))

---
## 6. Evaluasi Detail
Confusion matrix + classification report per kelas.

In [ ]:
model.load_state_dict(torch.load(RESULTS / f'{MODEL_NAME}_v3.pt', map_location=DEVICE))
model.to(DEVICE).eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, targets in tqdm(val_loader, desc='Evaluating'):
        inputs = inputs.to(DEVICE)
        outputs = model(inputs)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(targets.cpu().numpy().tolist())

print(classification_report(
    all_labels, all_preds,
    target_names=CLASS_LABELS,
    digits=4,
))

cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=[c.split('_', 1)[1] for c in CLASS_LABELS])
disp.plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix - Validation Set', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. History Plot
Train loss & val F1 per epoch.

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 4))

ax1.plot(result['history']['train_loss'], 'b-o', markersize=3, label='Train Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss', color='b')
ax1.tick_params(axis='y', labelcolor='b')

ax2 = ax1.twinx()
ax2.plot(result['history']['val_f1'], 'r-s', markersize=3, label='Val F1')
ax2.set_ylabel('F1 Macro', color='r')
ax2.tick_params(axis='y', labelcolor='r')

best_epoch = result['best_epoch']
ax2.axvline(best_epoch, color='gray', linestyle='--', alpha=0.5, label=f'Best @ E{best_epoch}')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.title('Training History', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 8. Save Final Info
Ringkasan hasil training.

In [ ]:
summary = {
    'model': MODEL_NAME,
    'batch_size': BATCH_SIZE,
    'transform': 'v3 (AugMix + RandomErasing)',
    'loss': 'ClassBalancedLoss (beta=0.999, gamma=2.0)',
    'mixup': USE_MIXUP,
    'sampler': 'WeightedRandomSampler',
    'device': DEVICE,
    'train_samples': len(train_df),
    'val_samples': len(val_df),
    'best_val_f1': result['best_val_f1'],
    'best_epoch': result['best_epoch'],
}

with open(RESULTS / f'{MODEL_NAME}_v3_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

for k, v in summary.items():
    print(f'{k}: {v}')
print(f'\nSaved: {RESULTS / f"{MODEL_NAME}_v3.pt"}')